In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
"""
app.py
------
Streamlit UI for Fake News Detection.

Models:
1. Linear SVM + TF-IDF
2. BiLSTM Deep Learning Model
3. DistilBERT Transformer Model

Run:
    streamlit run app/app.py
"""

import os
import sys
import streamlit as st

# ---------------------------------------------------------
# PATH SETUP
# ---------------------------------------------------------

APP_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(APP_DIR)
SRC_DIR = os.path.join(PROJECT_ROOT, "src")

sys.path.insert(0, SRC_DIR)

# Classical ML model
from predict import predict_news

# BiLSTM model
from predict_bilstm import predict_news_bilstm

# BERT / DistilBERT model
from predict_bert import predict_news_bert


# ---------------------------------------------------------
# PAGE CONFIGURATION
# ---------------------------------------------------------

st.set_page_config(
    page_title="Fake News Detection System",
    page_icon="📰",
    layout="centered"
)


# ---------------------------------------------------------
# CUSTOM CSS
# ---------------------------------------------------------

st.markdown(
    """
    <style>

    .main-title {
        font-size: 2.2rem;
        font-weight: 700;
        margin-bottom: 0.2rem;
    }

    .subtitle {
        color: #666;
        margin-bottom: 1.5rem;
    }

    .result-fake {
        background-color: #fdecea;
        border-left: 6px solid #d62728;
        padding: 1rem;
        border-radius: 6px;
        margin-bottom: 1rem;
    }

    .result-real {
        background-color: #eaf7ed;
        border-left: 6px solid #2ca02c;
        padding: 1rem;
        border-radius: 6px;
        margin-bottom: 1rem;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ---------------------------------------------------------
# TITLE
# ---------------------------------------------------------

st.markdown(
    '<div class="main-title">📰 Fake News Detection System</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'NLP + Machine Learning + Deep Learning'
    '</div>',
    unsafe_allow_html=True
)


# ---------------------------------------------------------
# INPUT
# ---------------------------------------------------------

news_text = st.text_area(
    "Enter a news article or headline:",
    height=220,
    placeholder="Paste a headline or full article text here..."
)


# ---------------------------------------------------------
# BUTTON
# ---------------------------------------------------------

check_clicked = st.button(
    " Check News",
    type="primary",
    use_container_width=True
)


# ---------------------------------------------------------
# PREDICTION
# ---------------------------------------------------------

if check_clicked:

    if not news_text.strip():

        st.warning("Please enter some text before checking.")

    else:

        # =================================================
        # CLASSICAL MODEL
        # =================================================

        st.subheader("Classical Machine Learning Model")

        try:

            with st.spinner("Running SVM + TF-IDF..."):

                svm_result = predict_news(news_text)

            svm_label = svm_result["label"]

            if svm_label == "FAKE":
                st.markdown(
                    '<div class="result-fake">',
                    unsafe_allow_html=True
                )
            else:
                st.markdown(
                    '<div class="result-real">',
                    unsafe_allow_html=True
                )

            st.markdown(
                f"### Prediction: {svm_label} NEWS"
            )

            st.write(
                f"**Confidence:** {svm_result['confidence']}%"
            )

            st.write(
                "**Model:** Linear SVM + TF-IDF"
            )

            if not svm_result["is_calibrated"]:
                st.caption(
                    "Confidence is a decision-score based measure, "
                    "not a calibrated probability."
                )

            st.markdown("</div>", unsafe_allow_html=True)

        except Exception as e:

            st.error(
                f"SVM model error: {e}"
            )


        # =================================================
        # BILSTM MODEL
        # =================================================

        st.subheader("BiLSTM Deep Learning Model")

        try:

            with st.spinner("Running BiLSTM..."):

                bilstm_result = predict_news_bilstm(news_text)

            bilstm_label = bilstm_result["label"]

            if bilstm_label == "FAKE":

                st.markdown(
                    '<div class="result-fake">',
                    unsafe_allow_html=True
                )

            else:

                st.markdown(
                    '<div class="result-real">',
                    unsafe_allow_html=True
                )

            st.markdown(
                f"### Prediction: {bilstm_label} NEWS"
            )

            st.write(
                f"**Confidence:** {bilstm_result['confidence']}%"
            )

            st.write(
                "**Model:** BiLSTM"
            )

            st.caption(
                "The confidence shown is the model's sigmoid output "
                "and is not a calibrated probability."
            )

            st.markdown(
                "</div>",
                unsafe_allow_html=True
            )

        except FileNotFoundError as e:

            st.error(
                f"BiLSTM artifacts not found: {e}"
            )

        except Exception as e:

            st.error(
                f"BiLSTM model error: {e}"
            )


        # =================================================
        # BERT / DISTILBERT MODEL
        # =================================================

        st.subheader("BERT Deep Learning Model")

        try:

            with st.spinner("Running DistilBERT..."):

                bert_result = predict_news_bert(news_text)

            bert_label = bert_result["label"]

            if bert_label == "FAKE":

                st.markdown(
                    '<div class="result-fake">',
                    unsafe_allow_html=True
                )

            else:

                st.markdown(
                    '<div class="result-real">',
                    unsafe_allow_html=True
                )

            st.markdown(
                f"### Prediction: {bert_label} NEWS"
            )

            st.write(
                f"**Confidence:** {bert_result['confidence']}%"
            )

            st.write(
                f"**Model:** {bert_result['model_used']}"
            )

            st.caption(
                "The confidence shown is the model's softmax output "
                "and is not a calibrated probability."
            )

            st.markdown(
                "</div>",
                unsafe_allow_html=True
            )

        except FileNotFoundError as e:

            st.error(
                f"BERT artifacts not found: {e}"
            )

        except Exception as e:

            st.error(
                f"BERT model error: {e}"
            )


        # =================================================
        # MODEL COMPARISON
        # =================================================

        if (
            "svm_result" in locals()
            and "bilstm_result" in locals()
            and "bert_result" in locals()
        ):

            st.divider()

            st.subheader(" Model Comparison")

            col1, col2, col3 = st.columns(3)

            with col1:

                st.metric(
                    "SVM Prediction",
                    svm_result["label"]
                )

                st.write(
                    f"Confidence: **{svm_result['confidence']}%**"
                )

            with col2:

                st.metric(
                    "BiLSTM Prediction",
                    bilstm_result["label"]
                )

                st.write(
                    f"Confidence: **{bilstm_result['confidence']}%**"
                )

            with col3:

                st.metric(
                    "BERT Prediction",
                    bert_result["label"]
                )

                st.write(
                    f"Confidence: **{bert_result['confidence']}%**"
                )

            # Agreement
            if (
                svm_result["label"] ==
                bilstm_result["label"] ==
                bert_result["label"]
            ):

                st.success(
                    f"✓ All three models predict "
                    f"**{svm_result['label']} NEWS**"
                )

            else:

                st.warning(
                    "⚠️ The models produced different predictions."
                )


# ---------------------------------------------------------
# FOOTER
# ---------------------------------------------------------

st.divider()

st.caption(
    "Academic project — Fake News Detection Using NLP, "
    "Machine Learning and Deep Learning."
)

st.caption(
    "Model predictions are statistical signals and should not "
    "be treated as definitive fact-checking."
)